# 00 — Data Understanding

**Group D — Grid Load Prediction**

Goal: understand the raw Uganda grid-load dataset before touching it. We load the raw file, inspect its shape and types, expose the corrupted cells, and quantify missingness per feature (**Assignment Task 1**).

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
REPO = Path.cwd()
while not (REPO / "src").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
import numpy as np
import pandas as pd
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print("Repo root:", REPO)

Repo root: /home/winzer/code/python/class/aion-gridload


## Load the raw data (untouched)
The loader maps known junk tokens (`N/A`, `unknown`, `###`, `?`, `-`) to NaN but leaves everything else exactly as it appears on disk.

In [2]:
from src.data_loading import load_dataset, RAW_DATA, TARGET
raw = load_dataset()
print('Source:', RAW_DATA.name)
print('Shape :', raw.shape)
raw.head(10)

Source: Uganda_Grid_Load_Dataset.csv
Shape : (1490, 11)


,Record_ID,Region,Hour,DayOfWeek,Temperature_C,Humidity_pct,Rainfall_mm,PopulationIndex,IndustrialIndex,SolarGenerationIndex,GridLoad_MW
0,1,Central,20,0,hot,52.0,3.1,93.0,89,0.03,850.3
1,2,Eastern,13,0,18.5,NaN,3.7,95.0,89,0.38,788.3
2,3,Western,14,4,22.7,90.0,NaN,71.0,55,0.13,548.0
3,4,Northern,10,0,19.6,41.0,0.1,NaN,78,0.45,604.6
4,5,Northern,12,0,27.4,88.0,0.0,86.0,high,0.70,546.1
5,6,Central,1,5,21.9,53.0,12.8,74.0,55,0.04,MW900
6,7,Central,11,1,24.3,48.0,0.0,88.0,41,0.49,599.1
7,8,Eastern,7,1,25.9,52.0,10.1,93.0,61,0.49,680.7
8,9,Eastern,1,1,32.0,86.0,1.4,86.0,60,0.02,661.0
9,10,Eastern,15,3,33.0,76.0,0.0,97.0,91,0.40,816.2


## Column types & first look
Note how columns that should be numeric (`Temperature_C`, `PopulationIndex`, ...) arrive as `object` because of text contamination like `"hot"`.

In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1490 entries, 0 to 1489
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Record_ID             1490 non-null   int64  
 1   Region                1490 non-null   str    
 2   Hour                  1490 non-null   int64  
 3   DayOfWeek             1490 non-null   int64  
 4   Temperature_C         1489 non-null   str    
 5   Humidity_pct          1489 non-null   float64
 6   Rainfall_mm           1487 non-null   float64
 7   PopulationIndex       1489 non-null   float64
 8   IndustrialIndex       1490 non-null   str    
 9   SolarGenerationIndex  1488 non-null   float64
 10  GridLoad_MW           1490 non-null   str    
dtypes: float64(4), int64(3), str(4)
memory usage: 128.2 KB


In [4]:
raw.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Record_ID,1490.0,NaN,NaN,NaN,740.146309,428.150445,1.0,369.25,739.5,1110.75,1480.0
Region,1490,4,Western,390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hour,1490.0,NaN,NaN,NaN,11.990604,7.079415,0.0,6.0,12.0,18.0,30.0
DayOfWeek,1490.0,NaN,NaN,NaN,3.053691,2.006655,0.0,1.0,3.0,5.0,6.0
Temperature_C,1489,172,32.8,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Humidity_pct,1489.0,NaN,NaN,NaN,64.055742,17.659648,-15.0,49.0,63.0,79.0,95.0
Rainfall_mm,1487.0,NaN,NaN,NaN,4.097445,3.985879,-12.0,0.0,3.3,6.95,18.4
PopulationIndex,1489.0,NaN,NaN,NaN,75.134318,14.548887,50.0,63.0,75.0,87.0,100.0
IndustrialIndex,1490,77,41,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SolarGenerationIndex,1488.0,NaN,NaN,NaN,0.245901,0.220371,0.0,0.07,0.18,0.37,1.8


## Expose corrupted / non-numeric cells
For each column we should be numeric, count cells that fail to parse as a number.

In [5]:
from src.data_loading import NUMERIC_COLS
bad = {}
for col in NUMERIC_COLS:
    if col in raw.columns:
        coerced = pd.to_numeric(raw[col], errors='coerce')
        # cells that were non-null originally but became NaN after coercion = junk text
        junk = raw[col].notna() & coerced.isna()
        bad[col] = int(junk.sum())
bad_series = pd.Series(bad, name='non_numeric_cells').sort_values(ascending=False)
bad_series

Temperature_C           1
GridLoad_MW             1
IndustrialIndex         1
Hour                    0
DayOfWeek               0
Rainfall_mm             0
Humidity_pct            0
PopulationIndex         0
SolarGenerationIndex    0
Name: non_numeric_cells, dtype: int64

In [6]:
# Show some example offending values
examples = {}
for col in NUMERIC_COLS:
    if col in raw.columns:
        coerced = pd.to_numeric(raw[col], errors='coerce')
        junk_vals = raw.loc[raw[col].notna() & coerced.isna(), col].unique()[:8]
        if len(junk_vals):
            examples[col] = list(junk_vals)
examples

{'Temperature_C': ['hot'],
 'IndustrialIndex': ['high'],
 'GridLoad_MW': ['MW900']}

## Task 1 — Missing values & percentage per feature
After coercing types, every junk token and blank counts as missing. We report count and percentage for each feature.

In [7]:
from src.preprocessing import coerce_types, missing_report
coerced = coerce_types(raw)
report = missing_report(coerced)
report

,missing_count,missing_pct,dtype
Rainfall_mm,3,0.20,float64
Temperature_C,2,0.13,float64
SolarGenerationIndex,2,0.13,float64
IndustrialIndex,1,0.07,float64
PopulationIndex,1,0.07,float64
GridLoad_MW,1,0.07,float64
Humidity_pct,1,0.07,float64
Hour,0,0.00,int64
Region,0,0.00,string
Record_ID,0,0.00,int64


In [8]:
print('Total cells missing :', int(coerced.isna().sum().sum()))
print('Rows with >=1 missing:', int(coerced.isna().any(axis=1).sum()), 'of', len(coerced))
print('Rows missing TARGET  :', int(coerced[TARGET].isna().sum()))

Total cells missing : 11
Rows with >=1 missing: 11 of 1490
Rows missing TARGET  : 1


## Inconsistent entries — out-of-range values
Beyond missing cells, some numbers are physically impossible (negative humidity, `Hour = 30`, `SolarGenerationIndex > 1`). These are data-entry errors and are treated as missing during cleaning, then imputed.

In [9]:
from src.preprocessing import range_violation_report
range_violation_report(coerced)

,valid_range,out_of_range
Hour,"[0, 23]",1
Humidity_pct,"[0, 100]",1
Rainfall_mm,"[0, 500]",1
SolarGenerationIndex,"[0, 1]",1
DayOfWeek,"[0, 6]",0
Temperature_C,"[-10, 55]",0
PopulationIndex,"[0, 100]",0
IndustrialIndex,"[0, 100]",0
GridLoad_MW,"[0, 5000]",0


## Target overview
`GridLoad_MW` is the value we will predict — a continuous regression target.

In [10]:
coerced[TARGET].describe()

count    1489.000000
mean      616.135930
std       102.302883
min       369.500000
25%       538.300000
50%       617.300000
75%       692.400000
max       881.000000
Name: GridLoad_MW, dtype: float64

### Takeaways
- Dataset is **cross-sectional tabular** (Region + Hour + DayOfWeek + weather/economic indices → grid load); no continuous timestamp, so this is regression, not classic forecasting.
- Several numeric columns are polluted with text (`hot`, `###`, `unknown`) and with physically-impossible values (negative humidity, `Hour=30`) → must coerce, range-check + impute.
- Missingness is spread across features (see table). Handling strategy is decided in `experiments/imputation_01_median_vs_knn.ipynb`.
- Next: cleaning + EDA in `01_eda.ipynb`.